### Synthèse du Code et de l'API Open-Meteo

#### 1. Rôle de l'API Open-Meteo (`/v1/archive`)
* **Récupération des données historiques :** Interroge l'API sur les 5 à 6 dernières années (2020–2025) pour une position géolocalisée (latitude/longitude). Précision à 9km.
* **Modélisation de l'irradiance inclinée (`global_tilted_irradiance`) :** L'API intègre en interne les équations de géométrie solaire (position du soleil heure par heure) pour calculer l'irradiance totale reçue par un panneau selon son inclinaison (`tilt`) et son orientation (`azimuth`), évitant ainsi le calcul manuel de l'angle d'incidence $\cos(i)$.

#### 2. Rôle du Code Python
* **Extraction & Structuration :** Récupère la réponse JSON de l'API et la convertit en `pandas.DataFrame`.
* **Agrégation Temporelle :** Regroupe les données par créneau horaire annuel (`Mois`, `Jour`, `Heure`) et calcule la moyenne de l'irradiance sur la période 2020–2025.
* **Gestion des Années Bissextiles :** Utilise l'année 2024 comme référence pour ordonner chronologiquement l'année complète du 1er janvier à 00h au 31 décembre à 23h tout en intégrant le 29 février.
* **Formatage de Sortie :** Renvoie un tableau final de **8 784 lignes** ($366 \text{ jours} \times 24 \text{ heures}$) exprimant la puissance surfacique moyenne en $\text{W/m}^2$ prête pour le calcul de production électrique.

In [38]:
import pandas as pd
import requests


def get_annual_tilted_irradiance_average(
    lat: float, lon: float, tilt: float = 30, azimuth: float = 0
) -> pd.DataFrame:
    
    """Calcule le profil moyen annuel heure par heure (366 jours x 24h) de l'irradiance

    captée par un panneau incliné/orienté, basé sur les données 2020-2025.

    :param lat: Latitude (ex: 48.8566)
    :param lon: Longitude (ex: 2.3522)
    :param tilt: Inclinaison du panneau en degrés (ex: 30, valeur la plus commune)
    :param azimuth: Orientation du panneau (0 = Sud, -90 = Est, 90 = Ouest) (orientation plein sud est la plus obtimale)
    :return: DataFrame avec la moyenne horaire sur 1 an (G_panneau_W_m2)
    """
    
    # 1. Requête API Open-Meteo avec la variable global_tilted_irradiance
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2020-01-01",
        "end_date": "2025-12-31",
        "hourly": ["global_tilted_irradiance"],
        "tilt": tilt,
        "azimuth": azimuth,
        "timezone": "Europe/Paris",
    }

    res = requests.get(url, params=params)
    res.raise_for_status()

    # 2. Construction du DataFrame
    df = pd.DataFrame(res.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    # 3. Extraction des composantes temporelles
    df["month"] = df["time"].dt.month
    df["day"] = df["time"].dt.day
    df["hour"] = df["time"].dt.hour

    # 4. Calcul de la moyenne sur 2020-2025 pour chaque créneau (Mois, Jour, Heure)
    result_df = (
        df.groupby(["month", "day", "hour"])[["global_tilted_irradiance"]]
        .mean()
        .reset_index()
    )

    # 5. Tri chronologique sur l'année complète (utilisation de 2024 pour gérer le 29 février)
    result_df["temp_date"] = pd.to_datetime(
        result_df.assign(year=2024)[["year", "month", "day", "hour"]]
    )
    result_df = (
        result_df.sort_values("temp_date")
        .drop(columns=["temp_date"])
        .reset_index(drop=True)
    )

    # 6. Renommage des colonnes
    result_df.rename(
        columns={
            "month": "Mois",
            "day": "Jour",
            "hour": "Heure",
            "global_tilted_irradiance": "G_panneau_W_m2",
        },
        inplace=True,
    )

    #suppression de la data du 29 février
    mask = ((result_df["Mois"] == 2) & (result_df["Jour"] == 29)) == False     
    result_df = result_df[mask]

    #modification du tableau pour correspondre au format Cairn
    Misenforme_df = result_df[["Heure","G_panneau_W_m2"]] 
    Misenforme_df["Heure"] = Misenforme_df["Heure"]*3600
    Misenforme_df["G_panneau_W_m2"] = Misenforme_df["G_panneau_W_m2"]/(10**6)
    Misenforme_df = Misenforme_df.rename(columns={"Heure": "Secondes"})
    Misenforme_df = Misenforme_df.rename(columns={"G_panneau_W_m2": "G_panneau_MW_m2"})

    #création des 4 premières colonnes
    entête = {          
    "Secondes": ["Time","01/01/2023  00:00:00", "s", "true"], 
    "G_panneau_MW_m2": ["PV.ProdElecParM2","Production PV par m2 de panneau centre France", "MW/m2", "true"],
    }

    entête_df = pd.DataFrame(entête)

    Misenforme_df = pd.concat((entête_df,Misenforme_df), ignore_index=True)
    

    


    
    return Misenforme_df
#    return result_df[1400:1442]
#    return mask[1400:1445]

In [39]:
# Exemple d'appel pour un panneau incliné à 30° et orienté Plein Sud (azimuth=0) à Paris :
get_annual_tilted_irradiance_average(lat=48.8566, lon=2.3522, tilt=30, azimuth=0)

,Secondes,G_panneau_MW_m2
0,Time,PV.ProdElecParM2
1,01/01/2023 00:00:00,Production PV par m2 de panneau centre France
2,s,MW/m2
3,true,true
4,0,0.0
...,...,...
8759,68400,0.0
8760,72000,0.0
8761,75600,0.0
8762,79200,0.0


In [5]:
# Exemple d'appel pour un panneau incliné à 30° et orienté Plein Sud (azimuth=0) à Paris :
get_annual_tilted_irradiance_average(lat=48.8566, lon=2.3522, tilt=30, azimuth=45)

,Secondes,G_panneau_MW_m2
0,Time,PV.ProdElecParM2
1,01/01/2023 00:00:00,Production PV par m2 de panneau centre France
2,s,MW/m2
3,true,true
4,0,0.0
...,...,...
8783,68400,0.0
8784,72000,0.0
8785,75600,0.0
8786,79200,0.0
